[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/09_docente_ensamble.ipynb)

# MLY1101 · Machine Learning — Actividad 3.2
## Modelos de ensamble

**Resultado de aprendizaje (RA3):** elabora soluciones avanzadas de aprendizaje automático
mediante la optimización de hiperparámetros, técnicas de ensamble y validación cruzada, para
garantizar la precisión y generalización del modelo frente a objetivos de negocio complejos.

**Indicador de logro (IL 3.2):** desarrolla modelos basados en técnicas de ensamble para mitigar
problemas de sesgo y varianza en escenarios de negocio complejos.

---

### De dónde viene la pregunta

La Actividad 3.1 terminó con un resultado incómodo: ajustar hiperparámetros **no mejoró nada**.
La reacción natural es *"entonces probemos un modelo más potente"*.

Eso es exactamente lo que hacemos hoy. Y también lo vamos a medir.

---

### Sesgo y varianza, en una tabla

El error de un modelo se descompone en dos partes que se combaten de forma distinta:

| | **Sesgo** | **Varianza** |
|---|---|---|
| Qué es | El modelo es demasiado simple para el problema | El modelo cambia mucho según los datos que le tocaron |
| Cómo se ve | Falla igual en entrenamiento y en prueba | Va perfecto en entrenamiento y mal en prueba |
| Ejemplo | Una recta para separar algo curvo | Un árbol sin límite de profundidad |
| Cómo se reduce | Modelo más flexible, mejores variables | **Promediar modelos**, más datos, regularizar |

**Los ensambles atacan sobre todo la varianza.** Promediar modelos que se equivocan en cosas
distintas cancela parte del error.

**Y no arreglan el sesgo.** Si todos los modelos comparten el mismo punto ciego —porque las
variables no contienen la información que hace falta—, promediarlos no lo elimina. Diez modelos
mirando por la misma ventana no ven más.

---

### Las tres familias

| Familia | Idea | Ejemplo |
|---|---|---|
| **Bagging** | Entrenar en paralelo sobre muestras distintas y promediar | Bosque aleatorio |
| **Boosting** | Entrenar en serie: cada modelo corrige los errores del anterior | Gradient boosting |
| **Votación / apilamiento** | Combinar modelos **distintos entre sí** | `VotingClassifier` |

> **Detalle que suele pasar desapercibido:** el bosque aleatorio que llevas usando desde la
> Actividad 2.2 **ya es un ensamble**. Son 200 árboles votando. No vamos a introducir los
> ensambles: llevamos toda la asignatura usando uno.

---

### Al final de la sesión debes entregar

La comparación de modelos con **métrica y costo juntos**, y una respuesta argumentada a: *¿el
ensamble más complejo gana lo suficiente para justificar lo que cuesta?*

> ### 🎓 Pauta docente — Actividad 3.2
>
> **6 horas pedagógicas**; ~3 h de trabajo guiado y el resto sobre el caso oficial del equipo.
>
> | Bloque | Min | Foco |
> |---|---|---|
> | 0 · Encuadre | 20 | Sesgo y varianza; el bosque ya era un ensamble |
> | 1 · El catálogo de candidatos ⭐ | 30 | Sin baseline no hay comparación |
> | 2 · Comparar con validación cruzada | 45 | Métrica **y** tiempo |
> | 3 · El ensamble por votación ⭐⭐ | 45 | **No mejora, y cuesta más** |
> | 4 · Sesgo o varianza | 25 | Diagnosticar cuál es el problema aquí |
> | Cierre | 15 | Informe |
>
> **El bloque 3 es el que sostiene la sesión.** El ensamble por votación **no se distingue** del
> mejor modelo solo (0,5938 contra 0,594) y encima tarda más. No lo adelantes.

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

import waymo
RUTA_DATOS = waymo.exigir_detecciones_reales(RAIZ)
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"
print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.supervisado import nodes as supervisado
from kedro_mly1101.pipelines.optimizacion import nodes as optimizacion

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
CONFIG, FUGA, AJUSTE = PARAMETROS["modelo"], PARAMETROS["fuga"], PARAMETROS["ajuste"]

# La misma cadena de siempre: limpieza del RA1 -> partición del RA2.
crudo = pd.read_parquet(RUTA_DATOS)
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

marcada = supervisado.particionar(
    supervisado.preparar_variables(limpio, CONFIG, FUGA), CONFIG
)
entrena = marcada[marcada["particion"] == "entrenamiento"]

print(f"Entrenamiento: {len(entrena):,} filas en {entrena[CONFIG['grupo']].nunique()} segmentos")
print(f"Métrica de trabajo: {AJUSTE['metrica']}  ·  pliegues: {AJUSTE['n_pliegues']}")

---
# Bloque 1 · ⭐ Contra qué se compara

Un resultado suelto no significa nada. Antes de comparar modelos complejos hace falta el piso:

| Candidato | Por qué está en la lista |
|---|---|
| **Baseline** | Responde siempre la clase mayoritaria. Si tu modelo no le gana, no hay modelo |
| **Árbol de decisión** | El más simple que aprende algo. Interpretable: se puede dibujar |
| **Regresión logística** | El modelo lineal. Si gana, el problema era lineal y sobraba lo demás |
| **Bosque aleatorio** | Bagging. El que venimos usando |
| **Gradient boosting** | Boosting. Corrige errores en serie |
| **Votación** | Combina árbol + bosque + boosting |

### ✏️ TODO 1 — Ejecutar la comparación

`optimizacion.comparar_ensambles()` evalúa los seis con validación cruzada **por grupo** y
cronometra cada uno.

*(Tarda unos 15 segundos: son 30 entrenamientos.)*

In [ ]:
comparacion = optimizacion.comparar_ensambles(marcada, CONFIG, AJUSTE)
comparacion

### ✏️ TODO 2 — Lo primero que hay que mirar

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Cuánto le saca la **regresión logística** al baseline?
2. ¿Qué te dice eso sobre la naturaleza del problema?

> ### 🎓 Pauta docente — Bloque 1
>
> **Cifras medidas** (Perception v2, 2026-09-08): gradient boosting **0,594** · ensamble
> **0,5938** · ruido **0,0282**. El detalle por pliegue sale de la celda; no recites la tabla
> del hilo viejo.
>
> **Respuesta al TODO 2:** si la logística queda pegada al baseline, el problema **no es
> linealmente separable**. Una frontera recta no distingue las difíciles. Los árboles, sí
> (aunque `LEVEL_2` siga en F1 0,0893).
>
> Merece decirse así:
>
> > *El modelo que peor funciona te dice algo sobre el problema, no solo sobre sí mismo. Que la
> > regresión logística fracase es información: la relación es no lineal.*
>
> **Fíjate también en la desviación del árbol: 0,0166**, el doble que la del bosque. Es varianza
> pura: un solo árbol depende mucho de qué datos le tocaron. El bosque, que promedia 200,
> estabiliza. **Ese contraste es la demostración de para qué sirve el bagging**, y está en la
> tabla antes de haber hablado del ensamble.
>
> **Criterio de logro:** compara contra el baseline **y** contra el modelo lineal, y extrae de
> ahí una conclusión sobre la naturaleza del problema.

---
# Bloque 2 · Métrica y costo, juntos

Fíjate en que la tabla trae una columna `segundos`. No es decoración.

La pregunta de esta actividad **no** es *"¿cuál saca el número más alto?"*. Es:

> **¿Gana lo suficiente para justificar lo que cuesta?**

Un modelo que gana 0,004 y tarda cuatro veces más no es mejor: es más caro.

### ✏️ TODO 3 — El costo relativo

In [ ]:
costo = comparacion.copy()
costo["veces_mas_lento"] = (costo["segundos"] / costo["segundos"].min()).round(1)
costo["ganancia_vs_arbol"] = (
    costo["media"] - costo.loc[costo["modelo"] == "arbol", "media"].iloc[0]
).round(4)
costo[["modelo", "media", "ganancia_vs_arbol", "segundos", "veces_mas_lento"]]

---
# Bloque 3 · ⭐⭐ El ensamble por votación

Ya tenemos tres modelos buenos. La intuición dice que combinarlos debería dar algo mejor que
cualquiera de los tres: cada uno se equivoca en cosas distintas y el voto cancela errores.

### ✏️ TODO 4 — Antes de mirar la tabla, apuesta

**Creo que el ensamble por votación quedará:** `____`
*(por encima del bosque / igual / por debajo)*

### ✏️ TODO 5 — La comparación directa

In [ ]:
individuales = comparacion[comparacion["modelo"] != "ensamble_votacion"]
mejor_solo = individuales.loc[individuales["media"].idxmax()]
ensamble = comparacion.loc[comparacion["modelo"] == "ensamble_votacion"].iloc[0]

diferencia = ensamble["media"] - mejor_solo["media"]
sobrecosto = ensamble["segundos"] / mejor_solo["segundos"]

print(f"Mejor individual      : {mejor_solo['modelo']} ({mejor_solo['media']:.4f})")
print(f"Ensamble              : {ensamble['media']:.4f}")
print(f"Diferencia en F1-macro : {diferencia:+.4f}")
print(f"Ruido entre pliegues   : {mejor_solo['desv_entre_pliegues']:.4f}")
print(f"Sobrecosto en tiempo   : {sobrecosto:.2f}×")

In [ ]:
# Autochequeo — v2: el ensamble no se distingue del mejor individual (GB).
assert abs(diferencia) < mejor_solo["desv_entre_pliegues"], (
    "la diferencia debería ser menor que el ruido entre pliegues"
)
print(f"✅ vs {mejor_solo['modelo']}: diferencia {diferencia:+.4f} < ruido {mejor_solo['desv_entre_pliegues']:.4f}")
print("   no hay evidencia de que el ensamble sea mejor.")
print(f"   Pero el ensamble tarda {sobrecosto:.1f}× lo que tarda el mejor solo.")
print()
print("   Mismo desempeño demostrable, más costo, menos interpretable.")

### ✏️ TODO 6 — Por qué no funcionó

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

El ensamble combina árbol, bosque y boosting. No mejoró. Da una explicación, sabiendo que
**los ensambles reducen varianza, no sesgo**.

*Pista: mira qué tienen en común los tres modelos combinados.*

> ### 🎓 Pauta docente — Bloque 3 ⭐⭐
>
> **Cifras medidas:**
>
> | | Búsqueda / GB | Ensamble |
> |---|---|---|
> | F1-macro (v2) | **0,594** | **0,5938** |
>
> **Diferencia: −0,0002**, menor que el ruido (**0,0282**). El ensamble no suma.
>
> ⚠️ **Los tiempos varían entre máquinas**; el ensamble **siempre** cuesta más que el más caro
> de sus miembros. Al corregir, mira el orden, no el porcentaje.
>
> **El TODO 4 funciona si apuestan.** Casi todos dicen "por encima": es lo que sugiere la
> intuición y lo que dicen los tutoriales. Queda por debajo.
>
> **Respuesta al TODO 6 — la explicación correcta.** Los tres modelos combinados son **todos de
> árboles**: árbol, bosque (árboles en paralelo) y boosting (árboles en serie). Trazan el mismo
> tipo de frontera y **se equivocan en las mismas detecciones**.
>
> Un ensamble funciona cuando sus miembros cometen errores **poco correlacionados**. Aquí están
> muy correlacionados, así que promediar no cancela casi nada. Y encima el árbol simple, que es
> el peor de los tres (0,5586), arrastra el promedio hacia abajo.
>
> > *Un ensamble no es "más modelos". Es más modelos **distintos**. Si todos miran por la misma
> > ventana, promediarlos no amplía la vista.*
>
> **Si alguien propone la mejora correcta**, reconócela: incluir la **regresión logística** en la
> votación aportaría un tipo de error distinto. En este caso probablemente empeoraría el promedio
> porque su desempeño es muy bajo (0,5056), pero el razonamiento es el bueno. Se puede probar en
> vivo si hay tiempo.
>
> **El otro remate**: el modelo que gana es el gradient boosting, no el bosque de la 2.2. Dos
> sesiones del RA3 —ajuste y ensamble— y el F1 de `LEVEL_2` sigue en **0,0893**. **Eso también
> es un resultado**, y saberlo con evidencia vale más que sospecharlo.
>
> **Criterio de logro:** identifica que la diferencia no supera el ruido, considera el costo, y
> explica el fracaso por la correlación entre los modelos combinados.

---
# Bloque 4 · ¿Sesgo o varianza?

Si ni el ajuste ni el ensamble mejoran, la pregunta es **qué limita** al modelo.

| Síntoma | Diagnóstico | Qué hacer |
|---|---|---|
| Va mucho mejor en entrenamiento que en validación | **Varianza** | Más datos, regularizar, promediar |
| Va parecido en ambos, y ambos mediocres | **Sesgo** | Mejores variables, modelo más flexible |

### ✏️ TODO 7 — El diagnóstico

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold, cross_val_score

X = entrena[CONFIG["variables"]]
y = entrena[CONFIG["objetivo"]]
grupos = entrena[CONFIG["grupo"]]

modelo = RandomForestClassifier(
    n_estimators=200, max_depth=12, class_weight="balanced",
    random_state=CONFIG["semilla"], n_jobs=-1,
)
modelo.fit(X, y)

en_entrenamiento = f1_score(y, modelo.predict(X), average="macro")
en_validacion = cross_val_score(
    modelo, X, y, groups=grupos,
    cv=GroupKFold(n_splits=AJUSTE["n_pliegues"]), scoring=AJUSTE["metrica"], n_jobs=-1,
).mean()

print(f"F1-macro en entrenamiento : {en_entrenamiento:.4f}")
print(f"F1-macro en validación    : {en_validacion:.4f}")
print(f"Brecha                    : {en_entrenamiento - en_validacion:.4f}")

**✍️ Tu respuesta al TODO 7:**

*(doble clic aquí y escribe)*

1. ¿El problema es sesgo o varianza?
2. Según ese diagnóstico, ¿qué habría que hacer para mejorar de verdad?
3. ¿Por qué eso explica que el ajuste y el ensamble no sirvieran?

> ### 🎓 Pauta docente — Bloque 4
>
> **Cifra medida:** con `max_depth=12` la brecha entre entrenamiento y validación es
> considerable —el bosque memoriza bastante—, pero **la validación se queda estancada en ~0,59
> para todos los modelos y todas las configuraciones probadas en las dos sesiones anteriores**.
>
> **Ese estancamiento es la clave del diagnóstico.** Si el problema fuera solo varianza,
> promediar (bosque) o regularizar (limitar profundidad) habría movido la validación. No la
> movió. **Hay un techo de sesgo**: la información necesaria para distinguir mejor las
> detecciones difíciles **no está en las siete variables** que le dimos.
>
> **Respuestas esperadas:**
>
> 1. **Ambas cosas, pero lo que limita es el sesgo.** La varianza existe y el bosque ya la
>    controla; el techo lo pone la información disponible.
> 2. **Mejores variables.** Lo que más se acerca: la distancia al sensor
>    (`√(x² + y²)` en vez de x e y por separado), el ángulo, el volumen de la caja, o
>    características del segmento. **Ingeniería de características, no más modelo.**
> 3. Porque el ajuste y el ensamble **atacan la varianza**, y la varianza no era el cuello de
>    botella. Es como afinar el motor cuando el problema es que falta camino.
>
> **La frase que cierra el RA3 hasta aquí:**
>
> > *Antes de probar un modelo más potente, pregúntate si el problema es que tu modelo no
> > aprende, o que tus datos no dicen.*
>
> **Si sobra tiempo**, el experimento es de dos minutos y es muy convincente: agregar
> `distancia = np.sqrt(box_center_x**2 + box_center_y**2)` como variable y reentrenar. Es la
> relación física que genera la etiqueta, y una sola variable derivada suele mover más que las
> dos sesiones anteriores juntas.
>
> **Criterio de logro:** diagnostica el techo de sesgo, propone ingeniería de características, y
> conecta ese diagnóstico con el fracaso del ajuste y del ensamble.

---
# Cierre · Informe de ensamble

### Los candidatos

| Modelo | Familia | F1-macro | Desv. | Segundos |
|---|---|---|---|---|
| `____` | | | | |
| `____` | | | | |
| `____` | | | | |

**Baseline:** `____` · **Mejor modelo individual:** `____` · **Ensamble:** `____`

### La comparación que importa

**Diferencia entre el ensamble y el mejor individual:** `____`
**Ruido entre pliegues:** `____`
**¿Es distinguible?** `____`
**Sobrecosto en tiempo:** `____`

**Decisión y por qué:** `____`

> Si eliges el más simple, **dilo con el argumento del costo y la interpretabilidad**, no como
> si te conformaras. Elegir el modelo suficiente es una decisión de ingeniería, no una renuncia.

### Diagnóstico

**¿Sesgo o varianza?** `____` · **Evidencia:** `____`
**Qué haría falta para mejorar de verdad:** `____`

> ### 🎓 Criterios de logro — Actividad 3.2 (IL 3.2)
>
> | Nivel | Descripción |
> |---|---|
> | **Destacado (4)** | Todo lo del 3, y además: explica el fracaso del ensamble por la **correlación entre sus miembros**; diagnostica el techo de sesgo con evidencia; propone una variable derivada concreta |
> | **Logrado (3)** | Compara los seis candidatos con métrica **y** costo; contrasta la diferencia contra el ruido; concluye que el ensamble no se justifica y lo argumenta |
> | **En desarrollo (2)** | Ejecuta la comparación y elige el de mayor media, sin considerar ruido ni costo |
> | **Inicial (1)** | No usa baseline, o concluye que el ensamble es mejor sin evidencia |
>
> **Lo primero al corregir:** si el informe elige el ensamble *"porque es más avanzado"*. Con
> estos datos es peor en las tres dimensiones —métrica, costo e interpretabilidad—, y esa es la
> lección de la sesión.